# Clustering

Vamos a implementar el modelo del paper [Multilingual E5 Text Embeddings: A Technical Report](https://arxiv.org/pdf/2402.05672) para construir embebimientos de cada tweet y luego visualizarlos.

In [2]:
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [3]:
# pip install transformers

In [4]:
# pip install tidyX

In [1]:
# Tokenizer
from transformers import AutoTokenizer, AutoModel

# Nerual Network Managment
import torch.nn.functional as F
from torch import Tensor

# Tweets Package manager
from tidyX import TextPreprocessor as tp

# Unsupervised Learning
from sklearn.cluster import KMeans

# Dimensionality reduction
from sklearn.manifold import TSNE

# File and OS managment
import os
import pickle

# Data Mandagment
import pandas as pd
import numpy as np

# progss bar
from tqdm import tqdm

# Data Visualization
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine
from rich import print

# Paths
path = r"/mnt/disk2/Data"
path_3_day = os.path.join(path,"3_Day_Graphs")
path_daily = os.path.join(path,"Daily_Graphs")

In [2]:
def average_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    """
    last_hidden_states: Last Hidden Layer of pretrained Model
    attention_mask: Attention mask of pretrained model
    """
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

In [3]:
tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-large')
model = AutoModel.from_pretrained('intfloat/multilingual-e5-large')

In [ ]:
# We load the tweets
archivos = os.listdir('/mnt/disk2/Data/Tweets_DataFrames/Tweets_Paro_Total')

# Open retweets edge list in any case
tweets_pre = (
    pd.read_pickle(os.path.join(path, "Tweets_DataFrames", "tweets_presample.gzip"), compression = "gzip")
    .rename(columns = {'ID': 'Tweet ID'})
    .loc[:,['Tweet ID', 'Author ID', 'Referenced Tweet Author ID', 'Text', 'Reference Type','Favorites']]
)

tweets_pre['Reference Type'].value_counts(dropna=False).apply(lambda x: f"{x:,}")

In [ ]:
tweets_pre.loc[tweets_pre['Reference Type'].isna(),'Reference Type'] = 'original_tweet'

pre_tweets_originales = tweets_pre.loc[tweets_pre['Reference Type']=='original_tweet',:]
pre_retweets = tweets_pre.loc[tweets_pre['Reference Type']=='retweeted',:]

# Cargar mapa de Silla + Congresistas
with open(os.path.join(path,"Pickle","User_Dicts","mapa_full.pkl"), "rb") as file:
    mapa = pickle.load(file)

pre_tweets_originales.loc[:,'Party'] = pre_tweets_originales.loc[:,"Author ID"].map(mapa)
pre_retweets.loc[:,'Party'] = pre_retweets.loc[:,"Referenced Tweet Author ID"].map(mapa)

print("Clasificación de Tweets originales según el Author ID")
print(pre_tweets_originales.value_counts('Party',dropna=False).apply(lambda x: f"{x:,}"))
print("Clasificación de Retweets siguiendo el criterio del testimonio\n(El retweet se clasifica según a quien se retweeta)")
print(pre_retweets.value_counts('Party',dropna=False).apply(lambda x: f"{x:,}"))

pre_tweets_originales_NPD = pre_tweets_originales.loc[pre_tweets_originales['Party'].isna(),:]
df_test = pre_tweets_originales_NPD.sample(n = 2_000)[['Tweet ID', 'Author ID', 'Text','Favorites']]
df_test.head(3)

In [ ]:
train_tweets_originales = (
    pre_tweets_originales
    .loc[pre_tweets_originales['Party'].notna(),['Tweet ID', 'Author ID', 'Text', 'Party','Favorites']]
)

train_retweets = (
    pre_retweets
    .loc[pre_retweets['Party'].notna(),['Tweet ID', 'Author ID', 'Text', 'Party','Favorites']]
)

df = pd.concat([train_tweets_originales,train_retweets])
df = df.sample(n = 5_000)[['Tweet ID', 'Author ID', 'Text','Party','Favorites']]
print(f"Número de tweets originales y retweets etiquetados {len(df):,}")
print(f"Número de tweets sin clasificación (testeo) {len(df_test):,}")
df.head()

In [7]:
# rts usuario
rts_usuario = pd.read_pickle('/mnt/disk2/Data/Pickle/User_Rts_Vector/rts_usuario_paro.pkl')

In [8]:
def top_tweets(rts_usuario:pd.DataFrame,df:pd.DataFrame,affiliation:str,n_twiteros=20,n_tweets=100, criterio = 'Favorites'):
    df = df[['Author ID', 'Text', criterio]]
    # Top Twitteros
    top_affiliation = (
        rts_usuario.sort_values([f'Retweets {affiliation}'], ascending = [False]).
        iloc[0:n_twiteros].
        index.tolist()
    )
    
    # Top Tweets Influyentes
    top_affiliation_tweets = (
        df.loc[df['Author ID'].isin(top_affiliation),['Text',criterio]]
        .sort_values(criterio,ascending=False)
        .iloc[0:n_tweets]
        .loc[:,'Text']
    )
    return top_affiliation_tweets

def preprocess_tweets(top_affiliation_tweets:pd.Series):
      
    # Each input text should start with "query: " or "passage: ", even for non-English texts.
    # For tasks other than retrieval, you can simply use the "query: " prefix.
    tqdm.pandas()
    top_affiliation_tweets = top_affiliation_tweets.progress_apply(lambda x: tp.preprocess(str(x)))
    top_affiliation_tweets = "passage: " + top_affiliation_tweets
    input_texts = top_affiliation_tweets.values
    texts_index = top_affiliation_tweets.index
    
    return input_texts.tolist(),list(texts_index)

In [9]:
# Tokenize the input texts
def embed_tweets(input_texts:list)-> Tensor:

    # Tokenize
    batch_dict = tokenizer(input_texts, max_length = 512, padding = True, truncation = True, return_tensors = 'pt')
    
    # get pretrained model output and average with attention mask
    outputs = model(**batch_dict)
    embeddings = average_pool(outputs.last_hidden_state, batch_dict['attention_mask'])

    # normalize embeddings
    embeddings = F.normalize(embeddings, p=2, dim=1)
    return embeddings

In [ ]:
# Extraer Tweets de Partido Político
embeddings_political = {}

for affiliation in ['Derecha', 'Izquierda', 'Centro']:
    top_affiliation_tweets = top_tweets(rts_usuario,df,affiliation)

    # Preprocesar tweets
    input_texts,_ = preprocess_tweets(top_affiliation_tweets)

    print(f"Cantidad de Tweets {len(input_texts)}")
    print(f"Primer Tweet de {affiliation} -> {input_texts[0]}")

    # Crear embeddings
    embeddings = embed_tweets(input_texts)
    
    embeddings_political[affiliation] = embeddings
    del embeddings

# GUardar el pickle para ahorrar memoria
with open("./embeddings_political.pkl","wb") as file:
    pickle.dump(embeddings_political,file)

In [8]:
with open("./embeddings_political.pkl","rb") as file:
    embeddings_political = pickle.load(file)

In [ ]:
embeddings_political

In [12]:
def political_score(political_tweets: dict, tweet_embeddings: np.ndarray, tweets_ids:list):

    scores = pd.DataFrame(columns=political_tweets.keys(), index=tweets_ids)

    # Para cada partido Politico
    for affiliation in political_tweets.keys():
        
        # Para cada tweet al cual se le va a calcular el "political score"
        for i,idx in enumerate(tweets_ids):
            scores_list = [] # Aquí se guardan las distancias del tweet a los 100 tweets referentes de cada ideología
            for j in range(len(political_tweets[affiliation])):
                
                political_tweet = political_tweets[affiliation][j].detach().numpy()
                tweet = tweet_embeddings[i].detach().numpy()
                
                # Almacenar distancia coseno
                scores_list.append(cosine(political_tweet, tweet))

            # Almacenar la media de las distancias coseno del tweet a la idología
            scores.loc[idx,affiliation] = np.mean(scores_list)
    
    # Especificar el Tweet ID como índice para poder consultar el texto original
    scores.index = tweets_ids
    return scores

In [51]:
df.iloc[0:100_000].to_pickle('~/df_export.pkl.gzip', compression='gzip')

In [23]:
XD = temp.apply(lambda x: scipy.special.softmax(list(x)),axis=1)

In [31]:
copia = temp.copy()

In [43]:
df_2 = pd.DataFrame(XD.to_list(),columns = ['Derecha', 'Izquierda', 'Centro'], index=XD.index)

In [ ]:
df_2.index[0]

In [ ]:
df[df['ID'] == df_2.index[0]]['Text'].iloc[0]

In [16]:
df = df.reset_index()

In [ ]:
df

In [ ]:
len(list(range(0,len(df),64)))

In [ ]:
scores = pd.DataFrame()
for i in tqdm(range(0,len(df),64), leave = False, total = len(list(range(0,len(df),64)))):
    # Obtenemos el texto de los tweets a analizar
    tweets_to_process = df.loc[i:i+63,('Tweet ID','Text')].set_index('Tweet ID')['Text']
    
    # preprocesamiento del texto
    input_texts, tweets_ids = preprocess_tweets(tweets_to_process)
    
    # Crear embeddings
    embeddings = embed_tweets(input_texts)
    
    # Calcular la tabla de "political score"
    temp = political_score(embeddings_political,embeddings,tweets_ids)
    
    # Concatenar tablas de forma iterativa
    scores = pd.concat([scores,temp])
    
scores

In [ ]:
scores = pd.DataFrame()
for i in tqdm(range(0,len(df),64)):
    # Obtenemos el texto de los tweets a analizar
    tweets_to_process = df.loc[i:i+63,('ID','Text')].set_index('ID')['Text']
    
    # preprocesamiento del texto
    input_texts, tweets_ids = preprocess_tweets(tweets_to_process)
    
    # Crear embeddings
    embeddings = embed_tweets(input_texts)
    
    # Calcular la tabla de "political score"
    temp = political_score(embeddings_political,embeddings,tweets_ids)
    
    # Concatenar tablas de forma iterativa
    scores = pd.concat([scores,temp])
    
scores

In [201]:
def clustering(embeddings:Tensor, n_clusters:int, random_state = 666)-> tuple:
    
    kmeans = KMeans(n_clusters = n_clusters, random_state = random_state)
    kmeans.fit(embeddings.detach().numpy())

    cluster_labels = kmeans.labels_
    centroids = kmeans.cluster_centers_
    return cluster_labels, centroids

In [ ]:
# Dimensionality reduction
def dimensionality_visualization(embeddings, cluster_labels, n_components):
    
    tsne = TSNE(n_components = n_components, random_state = 666)
    embeddings_2d = tsne.fit_transform(embeddings.detach().numpy())
    
    plt.figure(figsize = (8, 6))
    scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c = cluster_labels, cmap = 'viridis', alpha = 0.6)
    plt.colorbar(scatter)
    plt.title('Clustering of Tweets with t-SNE')
    plt.xlabel('t-SNE feature 1')
    plt.ylabel('t-SNE feature 2')
    plt.show()
    
dimensionality_visualization(embeddings, cluster_labels, 2)